In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import IEEE_IDENTITY, IEEE_TRANSACTION, REPORTS_EXPLORATION

REPORTS_EXPLORATION.mkdir(parents=True, exist_ok=True)
OUT = REPORTS_EXPLORATION

In [6]:
# Mettre un entier (ex. 100_000) pour tester sur un echantillon si la RAM est limitee
NROWS = None


def titre(txt):
    print("\n" + "=" * 70)
    print(txt)
    print("=" * 70)


def charger(path, nrows=NROWS):
    """Charge un CSV et reduit la memoire (float64 -> float32, int64 -> int32)."""
    df = pd.read_csv(path, nrows=nrows, low_memory=False)
    for col in df.select_dtypes("float64").columns:
        df[col] = df[col].astype("float32")
    for col in df.select_dtypes("int64").columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    return df


def rapport_manquants(df, nom):
    miss = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
    miss.rename("pct_manquant").to_csv(OUT / f"missing_{nom}.csv")
    print(f"\nColonnes avec >50 % de valeurs manquantes : {(miss > 50).sum()} / {df.shape[1]}")
    print("Top 10 :")
    print(miss.head(10).to_string())
    return miss


def fraude_par(df, col, top=15):
    t = (
        df.groupby(col, dropna=False)["isFraud"]
        .agg(nb_transactions="count", nb_fraudes="sum", taux_fraude="mean")
        .sort_values("nb_transactions", ascending=False)
        .head(top)
    )
    t["taux_fraude"] = (t["taux_fraude"] * 100).round(2)
    t.to_csv(OUT / f"fraud_by_{col}.csv")
    print(f"\nFraude par {col} :")
    print(t.to_string())
    return t

In [7]:
# ----------------------------------------------------------------------
# 1. Chargement
# ----------------------------------------------------------------------
titre("1. CHARGEMENT")
tr = charger(IEEE_TRANSACTION)
idt = charger(IEEE_IDENTITY)
print(f"Transactions : {tr.shape[0]:,} lignes x {tr.shape[1]} colonnes "
      f"({tr.memory_usage(deep=True).sum() / 1e6:.0f} Mo en memoire)")
print(f"Identites    : {idt.shape[0]:,} lignes x {idt.shape[1]} colonnes")



1. CHARGEMENT
Transactions : 590,540 lignes x 394 colonnes (1262 Mo en memoire)
Identites    : 144,233 lignes x 41 colonnes


In [8]:
# ----------------------------------------------------------------------
# 2. Qualite : doublons, valeurs manquantes
# ----------------------------------------------------------------------
titre("2. QUALITE DES DONNEES")
dup_tr = tr["TransactionID"].duplicated().sum()
dup_id = idt["TransactionID"].duplicated().sum()
print(f"Doublons TransactionID : transaction={dup_tr}, identity={dup_id}")

print("\n--- Valeurs manquantes : transactions ---")
miss_tr = rapport_manquants(tr, "transaction")
print("\n--- Valeurs manquantes : identite ---")
miss_id = rapport_manquants(idt, "identity")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(miss_tr.values, bins=20, color="#1f4e9c")
ax.set_xlabel("% de valeurs manquantes")
ax.set_ylabel("Nombre de colonnes")
ax.set_title("Distribution des valeurs manquantes (transactions)")
fig.tight_layout()
fig.savefig(OUT / "missing_distribution.png", dpi=150)
plt.close(fig)


2. QUALITE DES DONNEES
Doublons TransactionID : transaction=0, identity=0

--- Valeurs manquantes : transactions ---

Colonnes avec >50 % de valeurs manquantes : 174 / 394
Top 10 :
dist2    93.63
D7       93.41
D13      89.51
D14      89.47
D12      89.04
D6       87.61
D9       87.31
D8       87.31
V153     86.12
V140     86.12

--- Valeurs manquantes : identite ---

Colonnes avec >50 % de valeurs manquantes : 12 / 41
Top 10 :
id_24    96.71
id_25    96.44
id_07    96.43
id_08    96.43
id_21    96.42
id_22    96.42
id_23    96.42
id_27    96.42
id_26    96.42
id_18    68.72


In [9]:
# ----------------------------------------------------------------------
# 3. Jointure transaction <-> identite
# ----------------------------------------------------------------------
titre("3. COUVERTURE DE LA JOINTURE")
df = tr.merge(idt, on="TransactionID", how="left", indicator=True)
couverture = (df["_merge"] == "both").mean() * 100
print(f"Transactions avec identite associee : {couverture:.1f} %")
df["a_identite"] = (df["_merge"] == "both").astype(int)
print("\nTaux de fraude selon la presence d'une identite :")
print((df.groupby("a_identite")["isFraud"].mean() * 100).round(2).to_string())


3. COUVERTURE DE LA JOINTURE


C:\Users\Nour\AppData\Local\Temp\ipykernel_24872\3074563155.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = tr.merge(idt, on="TransactionID", how="left", indicator=True)


Transactions avec identite associee : 24.4 %

Taux de fraude selon la presence d'une identite :
a_identite
0    2.09
1    7.85


C:\Users\Nour\AppData\Local\Temp\ipykernel_24872\3074563155.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = tr.merge(idt, on="TransactionID", how="left", indicator=True)
C:\Users\Nour\AppData\Local\Temp\ipykernel_24872\3074563155.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["a_identite"] = (df["_merge"] == "both").astype(int)


In [10]:
# ----------------------------------------------------------------------
# 4. Desequilibre des classes
# ----------------------------------------------------------------------
titre("4. TAUX DE FRAUDE")
taux = df["isFraud"].mean() * 100
nb_f = int(df["isFraud"].sum())
print(f"Fraudes : {nb_f:,} / {len(df):,} = {taux:.2f} %")
print(f"Ratio normal/fraude : {(len(df) - nb_f) / nb_f:.1f} pour 1")

fig, ax = plt.subplots(figsize=(5, 4))
df["isFraud"].value_counts().sort_index().plot.bar(ax=ax, color=["#1f4e9c", "#c0392b"])
ax.set_xticklabels(["Legitime", "Fraude"], rotation=0)
ax.set_title(f"Desequilibre des classes (fraude = {taux:.2f} %)")
fig.tight_layout()
fig.savefig(OUT / "class_imbalance.png", dpi=150)
plt.close(fig)


4. TAUX DE FRAUDE
Fraudes : 20,663 / 590,540 = 3.50 %
Ratio normal/fraude : 27.6 pour 1


In [11]:
# ----------------------------------------------------------------------
# 5. Montants
# ----------------------------------------------------------------------
titre("5. MONTANTS (TransactionAmt)")
print(df.groupby("isFraud")["TransactionAmt"].describe().round(2).to_string())
print(f"\nMontants <= 0 : {(df['TransactionAmt'] <= 0).sum()}")

fig, ax = plt.subplots(figsize=(9, 4))
for val, lab, c in [(0, "Legitime", "#1f4e9c"), (1, "Fraude", "#c0392b")]:
    ax.hist(np.log1p(df.loc[df["isFraud"] == val, "TransactionAmt"]),
            bins=60, alpha=0.6, density=True, label=lab, color=c)
ax.set_xlabel("log(1 + montant)")
ax.set_title("Distribution des montants")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "amount_distribution.png", dpi=150)
plt.close(fig)


5. MONTANTS (TransactionAmt)
            count    mean     std   min    25%   50%    75%       max
isFraud                                                              
0        569877.0  134.51  239.40  0.25  43.97  68.5  120.0  31937.39
1         20663.0  149.24  232.21  0.29  35.04  75.0  161.0   5191.00

Montants <= 0 : 0


In [12]:
# ----------------------------------------------------------------------
# 6. Temps (TransactionDT = secondes depuis une origine inconnue)
# ----------------------------------------------------------------------
titre("6. DIMENSION TEMPS (TransactionDT)")
duree_jours = (df["TransactionDT"].max() - df["TransactionDT"].min()) / 86400
print(f"Periode couverte : {duree_jours:.0f} jours")
DATE_DEPART = pd.Timestamp("2017-12-01")  # hypothese a documenter dans le rapport
df["datetime"] = DATE_DEPART + pd.to_timedelta(df["TransactionDT"], unit="s")
df["heure"] = df["datetime"].dt.hour
df["jour_semaine"] = df["datetime"].dt.dayofweek
print(f"Plage reconstruite : {df['datetime'].min()} -> {df['datetime'].max()}")

fraud_h = df.groupby("heure")["isFraud"].mean() * 100
print("\nTaux de fraude par heure (%) :")
print(fraud_h.round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
fraud_h.plot.bar(ax=ax, color="#1f4e9c")
ax.set_ylabel("Taux de fraude (%)")
ax.set_xlabel("Heure")
ax.set_title("Taux de fraude par heure (date de depart fictive)")
fig.tight_layout()
fig.savefig(OUT / "fraud_by_hour.png", dpi=150)
plt.close(fig)


6. DIMENSION TEMPS (TransactionDT)
Periode couverte : 182 jours
Plage reconstruite : 2017-12-02 00:00:00 -> 2018-06-01 23:58:51

Taux de fraude par heure (%) :
heure
0      3.14
1      3.13
2      3.75
3      3.83
4      5.19
5      7.03
6      7.77
7     10.61
8      9.30
9      9.00
10     5.32
11     3.88
12     3.04
13     2.29
14     2.42
15     2.54
16     2.95
17     3.15
18     3.52
19     3.47
20     3.43
21     3.40
22     3.27
23     3.70


C:\Users\Nour\AppData\Local\Temp\ipykernel_24872\4141588814.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["datetime"] = DATE_DEPART + pd.to_timedelta(df["TransactionDT"], unit="s")
C:\Users\Nour\AppData\Local\Temp\ipykernel_24872\4141588814.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["heure"] = df["datetime"].dt.hour
C:\Users\Nour\AppData\Local\Temp\ipykernel_24872\4141588814.py:10: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  

In [13]:
# ----------------------------------------------------------------------
# 7. Fraude par variable categorielle
# ----------------------------------------------------------------------
titre("7. FRAUDE PAR CATEGORIE")
for col in ["ProductCD", "card4", "card6", "DeviceType", "P_emaildomain"]:
    if col in df.columns:
        fraude_par(df, col)


7. FRAUDE PAR CATEGORIE

Fraude par ProductCD :
           nb_transactions  nb_fraudes  taux_fraude
ProductCD                                          
W                   439670        8969         2.04
C                    68519        8008        11.69
R                    37699        1426         3.78
H                    33024        1574         4.77
S                    11628         686         5.90

Fraude par card4 :
                  nb_transactions  nb_fraudes  taux_fraude
card4                                                     
visa                       384767       13373         3.48
mastercard                 189217        6496         3.43
american express             8328         239         2.87
discover                     6651         514         7.73
NaN                          1577          41         2.60

Fraude par card6 :
                 nb_transactions  nb_fraudes  taux_fraude
card6                                                    
debit             

In [14]:
# ----------------------------------------------------------------------
# 8. Candidats pour un identifiant client
# ----------------------------------------------------------------------
titre("8. CANDIDATS POUR LA DIMENSION CLIENT")
for col in ["card1", "card2", "addr1", "P_emaildomain"]:
    print(f"{col:15s} valeurs distinctes = {df[col].nunique():>8,}  "
          f"manquantes = {df[col].isna().mean() * 100:5.1f} %")
df["client_cle"] = (
    df["card1"].astype(str) + "_" + df["addr1"].astype(str) + "_" + df["P_emaildomain"].astype(str)
)
print(f"\nCle composee card1+addr1+email : {df['client_cle'].nunique():,} clients potentiels")



8. CANDIDATS POUR LA DIMENSION CLIENT
card1           valeurs distinctes =   13,553  manquantes =   0.0 %
card2           valeurs distinctes =      500  manquantes =   1.5 %
addr1           valeurs distinctes =      332  manquantes =  11.1 %
P_emaildomain   valeurs distinctes =       59  manquantes =  16.0 %

Cle composee card1+addr1+email : 73,407 clients potentiels


C:\Users\Nour\AppData\Local\Temp\ipykernel_24872\2013915673.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["client_cle"] = (


In [15]:
# ----------------------------------------------------------------------
# 9. Fiche de sources (a copier dans le rapport)
# ----------------------------------------------------------------------
titre("9. FICHE DE SOURCES")
fiche = f"""# Fiche de sources : IEEE-CIS Fraud Detection

| Attribut | train_transaction.csv | train_identity.csv |
|---|---|---|
| Format | CSV | CSV |
| Lignes | {tr.shape[0]:,} | {idt.shape[0]:,} |
| Colonnes | {tr.shape[1]} | {idt.shape[1]} |
| Cle de jointure | TransactionID | TransactionID |
| Doublons sur la cle | {dup_tr} | {dup_id} |
| Colonnes > 50 % manquantes | {(miss_tr > 50).sum()} | {(miss_id > 50).sum()} |
| Proprietaire | Vesta Corporation (Kaggle) | Vesta Corporation (Kaggle) |
| Sensibilite RGPD | Anonymise | Anonymise |

## Indicateurs cles
- Taux de fraude : {taux:.2f} % ({nb_f:,} fraudes)
- Couverture de la jointure (transactions avec identite) : {couverture:.1f} %
- Periode couverte : {duree_jours:.0f} jours (date de depart fictive : {DATE_DEPART.date()})
- Montant moyen : {df['TransactionAmt'].mean():.2f} | median : {df['TransactionAmt'].median():.2f}

## Points d'attention
- Forte proportion de valeurs manquantes, surtout dans les colonnes id_* et V*.
- TransactionDT est un delta en secondes, pas une vraie date.
- Aucun identifiant client explicite : reconstruction a partir de card1, addr1, P_emaildomain.
"""
(OUT / "fiche_sources.md").write_text(fiche, encoding="utf-8")
print(fiche)
print(f"\nTous les resultats sont dans : {OUT}")



9. FICHE DE SOURCES
# Fiche de sources : IEEE-CIS Fraud Detection

| Attribut | train_transaction.csv | train_identity.csv |
|---|---|---|
| Format | CSV | CSV |
| Lignes | 590,540 | 144,233 |
| Colonnes | 394 | 41 |
| Cle de jointure | TransactionID | TransactionID |
| Doublons sur la cle | 0 | 0 |
| Colonnes > 50 % manquantes | 174 | 12 |
| Proprietaire | Vesta Corporation (Kaggle) | Vesta Corporation (Kaggle) |
| Sensibilite RGPD | Anonymise | Anonymise |

## Indicateurs cles
- Taux de fraude : 3.50 % (20,663 fraudes)
- Couverture de la jointure (transactions avec identite) : 24.4 %
- Periode couverte : 182 jours (date de depart fictive : 2017-12-01)
- Montant moyen : 135.03 | median : 68.77

## Points d'attention
- Forte proportion de valeurs manquantes, surtout dans les colonnes id_* et V*.
- TransactionDT est un delta en secondes, pas une vraie date.
- Aucun identifiant client explicite : reconstruction a partir de card1, addr1, P_emaildomain.


Tous les resultats sont dans : C: